# Gold — Analyse comptable
Pipeline Olist : indicateurs financiers à partir de la zone silver.

In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as _sum, avg, count, round as _round,
    countDistinct, date_format, max as _max,
    collect_set, concat_ws, create_map, lit, coalesce
)
from itertools import chain

spark = SparkSession.builder \
    .appName("olist-gold-account") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 21:25:39 WARN Utils: Your hostname, edilene-ThinkPad-P53, resolves to a loopback address: 127.0.1.1; using 192.168.1.89 instead (on interface wlp82s0)
26/06/17 21:25:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 21:25:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/17 21:25:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/17 21:25:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Chargement des tables silver

In [2]:
df_orders = spark.read.parquet("../data/silver/orders/")
df_customers = spark.read.parquet("../data/silver/customers/")
df_items = spark.read.parquet("../data/silver/order_items/")
df_payments = spark.read.parquet("../data/silver/payments/")
df_products = spark.read.parquet("../data/silver/products/")
df_sellers = spark.read.parquet("../data/silver/sellers/")
df_pcnt = spark.read.parquet("../data/silver/category_translation/")

## Segmentation des commandes par statut
Trois segments : revenu reconnu, en transit, exclu.

In [3]:
df_orders.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  625|
|    invoiced|  314|
|     created|    5|
|   delivered|96478|
| unavailable|  609|
|  processing|  301|
|    approved|    2|
+------------+-----+



In [4]:
# Trois segments selon le statut de la commande
df_orders_delivered = df_orders.filter(col("order_status") == "delivered")

df_orders_in_transit = df_orders.filter(
    col("order_status").isin(["created", "approved", "processing", "invoiced", "shipped"])
)

df_orders_excluded = df_orders.filter(
    col("order_status").isin(["canceled", "unavailable"])
)

## Chiffre d'affaires par segment
Jointure order_items + orders

In [5]:
df_gold_revenue_delivered = df_items.join(
    df_orders_delivered.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

df_gold_revenue_in_transit = df_items.join(
    df_orders_in_transit.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

df_gold_revenue_excluded = df_items.join(
    df_orders_excluded.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

In [6]:
# Fonction pour calculer le chiffre d'affaires d'un segment donné
def calculate_revenue(df, name):
    result = df.agg(
        _round(_sum("price"), 2).alias("ca_produits"),
        _round(_sum("freight_value"), 2).alias("total_frais_livraison"),
        _round(_sum("price") + _sum("freight_value"), 2).alias("ca_total")
    )
    print(f"=== {name} ===")
    result.show()
    return result

ca_delivered = calculate_revenue(df_gold_revenue_delivered, "Revenu reconnu (delivered)")
ca_in_transit = calculate_revenue(df_gold_revenue_in_transit, "Revenu en transit (payé, non livré)")
ca_excluded = calculate_revenue(df_gold_revenue_excluded, "Exclu (annulé/indisponible)")

=== Revenu reconnu (delivered) ===
+-------------+---------------------+-------------+
|  ca_produits|total_frais_livraison|     ca_total|
+-------------+---------------------+-------------+
|1.322149811E7|           2198275.64|1.541977375E7|
+-------------+---------------------+-------------+

=== Revenu en transit (payé, non livré) ===
+-----------+---------------------+---------+
|ca_produits|total_frais_livraison| ca_total|
+-----------+---------------------+---------+
|  272902.63|             42850.65|315753.28|
+-----------+---------------------+---------+

=== Exclu (annulé/indisponible) ===
+-----------+---------------------+---------+
|ca_produits|total_frais_livraison| ca_total|
+-----------+---------------------+---------+
|   97242.96|             10783.25|108026.21|
+-----------+---------------------+---------+



## Chiffre d'affaires mensuel
Évolution du revenu reconnu dans le temps.

In [7]:
df_monthly_revenue = df_gold_revenue_delivered.withColumn(
    "month", date_format(col("order_purchase_timestamp"), "yyyy-MM")
).groupBy("month").agg(
    _round(_sum("price"), 2).alias("ca_produits"),
    _round(_sum("freight_value"), 2).alias("total_frais_livraison"),
    _round(_sum("price") + _sum("freight_value"), 2).alias("ca_total")
).orderBy("month")

df_monthly_revenue.show(30)

+-------+-----------+---------------------+----------+
|  month|ca_produits|total_frais_livraison|  ca_total|
+-------+-----------+---------------------+----------+
|2016-09|     134.97|                 8.49|    143.46|
|2016-10|   40325.11|              6165.55|  46490.66|
|2016-12|       10.9|                 8.72|     19.62|
|2017-01|  111798.36|             15684.01| 127482.37|
|2017-02|   234223.4|             37015.92| 271239.32|
|2017-03|  359198.85|              55132.1| 414330.95|
|2017-04|  340669.68|             50142.72|  390812.4|
|2017-05|  489338.25|             77513.15|  566851.4|
|2017-06|  421923.37|              68127.0| 490050.37|
|2017-07|  481604.52|             84694.56| 566299.08|
|2017-08|   554699.7|             91132.66| 645832.36|
|2017-09|  607399.67|             93677.82| 701077.49|
|2017-10|  648247.65|            102869.36| 751117.01|
|2017-11|  987765.37|            165598.83| 1153364.2|
|2017-12|  726033.19|             117045.1| 843078.29|
|2018-01| 

## Validation — paiements par commande
Vérification de la cardinalité avant toute jointure avec payments.

In [8]:
# Combien de paiements par commande ?
payments_per_order = df_payments.groupBy("order_id").agg(count("*").alias("nb_payments"))
payments_per_order.groupBy("nb_payments").count().orderBy("nb_payments").show()

+-----------+-----+
|nb_payments|count|
+-----------+-----+
|          1|96479|
|          2| 2382|
|          3|  301|
|          4|  108|
|          5|   52|
|          6|   36|
|          7|   28|
|          8|   11|
|          9|    9|
|         10|    5|
|         11|    8|
|         12|    8|
|         13|    3|
|         14|    2|
|         15|    2|
|         19|    2|
|         21|    1|
|         22|    1|
|         26|    1|
|         29|    1|
+-----------+-----+



In [9]:
# Inspection d'un cas extrême (commande avec le plus de paiements)
example_order_id = df_payments.groupBy("order_id").count().orderBy(col("count").desc()).first()["order_id"]
df_payments.filter(col("order_id") == example_order_id).orderBy("payment_sequential").show(30)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|fa65dad1b0e818e3c...|                 1|     voucher|                   1|         3.71|
|fa65dad1b0e818e3c...|                 2|     voucher|                   1|         8.51|
|fa65dad1b0e818e3c...|                 3|     voucher|                   1|         2.95|
|fa65dad1b0e818e3c...|                 4|     voucher|                   1|        29.16|
|fa65dad1b0e818e3c...|                 5|     voucher|                   1|         0.66|
|fa65dad1b0e818e3c...|                 6|     voucher|                   1|         5.02|
|fa65dad1b0e818e3c...|                 7|     voucher|                   1|         0.32|
|fa65dad1b0e818e3c...|                 8|     voucher|                   1|        26.02|
|fa65dad1b

## Répartition des paiements par type
Valeur totale, nombre de transactions et part de chaque type.

In [10]:
total_payments = df_payments.agg(_sum("payment_value")).collect()[0][0]

df_gold_payment_breakdown = df_payments.groupBy("payment_type").agg(
    _round(_sum("payment_value"), 2).alias("valeur_totale"),
    count("*").alias("nb_transactions"),
    _round(avg("payment_value"), 2).alias("valeur_moyenne")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_payments) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_gold_payment_breakdown.show()

+------------+-------------+---------------+--------------+------------------+
|payment_type|valeur_totale|nb_transactions|valeur_moyenne|pourcentage_valeur|
+------------+-------------+---------------+--------------+------------------+
| credit_card|1.254208419E7|          76795|        163.32|             78.34|
|      boleto|   2869361.27|          19784|        145.03|             17.92|
|     voucher|    379436.87|           5775|          65.7|              2.37|
|  debit_card|    217989.79|           1529|        142.57|              1.36|
| not_defined|          0.0|              3|           0.0|               0.0|
+------------+-------------+---------------+--------------+------------------+



In [11]:
# Qualité des données : transactions voucher à valeur nulle
zero_value_vouchers = df_payments.filter(
    (col("payment_type") == "voucher") & (col("payment_value") == 0)
).count()
print(f"Transactions voucher avec valeur zéro : {zero_value_vouchers}")

Transactions voucher avec valeur zéro : 6


## Parcellement moyen par type de paiement

In [12]:
df_installments_by_type = df_payments.groupBy("payment_type").agg(
    _round(avg("payment_installments"), 2).alias("parcelles_moyennes"),
    count("*").alias("nb_transactions")
).orderBy(col("nb_transactions").desc())

df_installments_by_type.show()

+------------+------------------+---------------+
|payment_type|parcelles_moyennes|nb_transactions|
+------------+------------------+---------------+
| credit_card|              3.51|          76795|
|      boleto|               1.0|          19784|
|     voucher|               1.0|           5775|
|  debit_card|               1.0|           1529|
| not_defined|               1.0|              3|
+------------+------------------+---------------+



## Panier moyen par commande
Agrégation des paiements au niveau commande avant jointure.

In [16]:
df_payments_agg = df_payments.groupBy("order_id").agg(
    _sum("payment_value").alias("valeur_totale_payee"),
    _max("payment_installments").alias("max_parcelles"),
    concat_ws(",", collect_set("payment_type")).alias("types_paiement")
)

In [14]:
df_orders_for_payment_analysis = df_orders.filter(
    ~col("order_status").isin(["canceled", "unavailable"])
)

df_gold_payments = df_payments_agg.join(
    df_orders_for_payment_analysis.select("order_id", "customer_id", "order_status", "order_purchase_timestamp"),
    "order_id", "inner"
)

In [15]:
df_gold_payments.agg(
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).show()

df_gold_payments.groupBy("types_paiement").agg(
    count("*").alias("nb_commandes"),
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).orderBy(col("nb_commandes").desc()).show()

+------------+
|panier_moyen|
+------------+
|      160.27|
+------------+



+--------------------+------------+------------+
|      types_paiement|nb_commandes|panier_moyen|
+--------------------+------------+------------+
|         credit_card|       73407|      166.32|
|              boleto|       19539|      144.67|
| credit_card,voucher|        2210|       148.9|
|             voucher|        1535|      105.23|
|          debit_card|        1514|      140.27|
|credit_card,debit...|           1|      152.82|
+--------------------+------------+------------+



## Paiements par état du client (demande géographique)

In [18]:
total_customer_payments = df_payments_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "inner"
).agg(_sum("valeur_totale_payee")).collect()[0][0]

df_payments_by_customer_state = df_payments_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "inner"
).join(
    df_customers.select("customer_id", "customer_state"), "customer_id", "inner"
).groupBy("customer_state").agg(
    _round(_sum("valeur_totale_payee"), 2).alias("valeur_totale"),
    count("*").alias("nb_commandes"),
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_customer_payments) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_payments_by_customer_state.show(27)

+--------------+-------------+------------+------------+------------------+
|customer_state|valeur_totale|nb_commandes|panier_moyen|pourcentage_valeur|
+--------------+-------------+------------+------------+------------------+
|            SP|   5998226.96|       41745|      143.69|             37.47|
|            RJ|   2144379.69|       12852|      166.85|             13.39|
|            MG|   1872257.26|       11635|      160.92|              11.7|
|            RS|    890898.54|        5466|      162.99|              5.57|
|            PR|    811156.38|        5045|      160.78|              5.07|
|            SC|    623086.43|        3637|      171.32|              3.89|
|            BA|    616645.82|        3380|      182.44|              3.85|
|            DF|    355141.08|        2140|      165.95|              2.22|
|            GO|    350092.31|        2020|      173.31|              2.19|
|            ES|    325967.55|        2033|      160.34|              2.04|
|           

## Paiements par région du client

In [19]:
region_map = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

mapping_expr = create_map([lit(x) for x in chain(*region_map.items())])

df_payments_by_customer_region = df_payments_by_customer_state.withColumn(
    "region", mapping_expr[col("customer_state")]
).groupBy("region").agg(
    _round(_sum("valeur_totale"), 2).alias("valeur_totale"),
    _sum("nb_commandes").alias("nb_commandes")
).orderBy(col("valeur_totale").desc())

df_payments_by_customer_region.show()

+------------+-------------+------------+
|      region|valeur_totale|nb_commandes|
+------------+-------------+------------+
|     Sudeste|1.034083146E7|       68265|
|         Sul|   2325141.35|       14148|
|    Nordeste|   1898479.44|        9394|
|Centro-Oeste|   1029797.52|        5782|
|       Norte|    414622.35|        1851|
+------------+-------------+------------+



## Validation — vendeurs par commande

In [20]:
sellers_per_order = df_items.groupBy("order_id").agg(countDistinct("seller_id").alias("nb_sellers"))
sellers_per_order.groupBy("nb_sellers").count().orderBy("nb_sellers").show()

+----------+-----+
|nb_sellers|count|
+----------+-----+
|         1|97388|
|         2| 1219|
|         3|   54|
|         4|    3|
|         5|    2|
+----------+-----+



## Chiffre d'affaires par état du vendeur (offre géographique)

In [21]:
total_seller_revenue = df_items.agg(_sum("price")).collect()[0][0]

df_revenue_by_seller_state = df_items.join(
    df_sellers.select("seller_id", "seller_state"), "seller_id", "inner"
).groupBy("seller_state").agg(
    _round(_sum("price"), 2).alias("valeur_totale"),
    count("*").alias("nb_items"),
    _round(avg("price"), 2).alias("prix_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_seller_revenue) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_revenue_by_seller_state.show(30)

+------------+-------------+--------+----------+------------------+
|seller_state|valeur_totale|nb_items|prix_moyen|pourcentage_valeur|
+------------+-------------+--------+----------+------------------+
|          SP|   8753396.21|   80342|    108.95|              64.4|
|          PR|   1261887.21|    8671|    145.53|              9.28|
|          MG|   1011564.74|    8827|     114.6|              7.44|
|          RJ|    843984.22|    4818|    175.17|              6.21|
|          SC|    632426.07|    4075|     155.2|              4.65|
|          RS|    378559.54|    2199|    172.15|              2.79|
|          BA|    285561.56|     643|    444.11|               2.1|
|          DF|     97749.48|     899|    108.73|              0.72|
|          PE|     91493.85|     448|    204.23|              0.67|
|          GO|     66399.21|     520|    127.69|              0.49|
|          ES|     47689.61|     372|     128.2|              0.35|
|          MA|     36408.95|     405|      89.9|

## Top vendeurs par chiffre d'affaires généré
Basé sur le revenu reconnu (commandes livrées).

In [22]:
df_top_sellers = df_items.join(
    df_orders_delivered.select("order_id"), "order_id", "inner"
).join(
    df_sellers.select("seller_id", "seller_city", "seller_state"), "seller_id", "inner"
).groupBy("seller_id", "seller_city", "seller_state").agg(
    _round(_sum("price"), 2).alias("ca_genere"),
    count("*").alias("nb_items_vendus"),
    _round(avg("price"), 2).alias("prix_moyen")
).orderBy(col("ca_genere").desc())

df_top_sellers.show(20, truncate=False)

+--------------------------------+---------------------+------------+---------+---------------+----------+
|seller_id                       |seller_city          |seller_state|ca_genere|nb_items_vendus|prix_moyen|
+--------------------------------+---------------------+------------+---------+---------------+----------+
|4869f7a5dfa277a7dca6462dcf3b52b2|guariba              |SP          |226987.93|1148           |197.72    |
|53243585a1d6dc2643021fd1853d8905|lauro de freitas     |BA          |217940.44|400            |544.85    |
|4a3ca9315b744ce9f8e9374361493884|ibitinga             |SP          |196882.12|1949           |101.02    |
|fa1c13f2614d7b5c4749cbc52fecda94|sumare               |SP          |190917.14|579            |329.74    |
|7c67e1448b00f6e969d365cea6b010ab|itaquaquecetuba      |SP          |186570.05|1355           |137.69    |
|7e93a43ef30c4f03f38b393420bc753a|barueri              |SP          |165981.49|322            |515.47    |
|da8622b14eb17ae2831f4ac5b9dab84a|pir

## Chiffre d'affaires par catégorie de produit
`product_category_name` déjà nettoyé en silver (valeur 'unknown' si absente).

In [23]:
total_category_revenue = df_items.agg(_sum("price")).collect()[0][0]

df_revenue_by_category = df_items.join(
    df_products.select("product_id", "product_category_name"), "product_id", "left"
).join(
    df_pcnt.select("product_category_name", "product_category_name_english"),
    "product_category_name", "left"
).withColumn(
    "categorie",
    coalesce(col("product_category_name_english"), col("product_category_name"))
).groupBy("categorie").agg(
    _round(_sum("price"), 2).alias("valeur_totale"),
    count("*").alias("nb_items"),
    _round(avg("price"), 2).alias("prix_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_category_revenue) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_revenue_by_category.show(30)

+--------------------+-------------+--------+----------+------------------+
|           categorie|valeur_totale|nb_items|prix_moyen|pourcentage_valeur|
+--------------------+-------------+--------+----------+------------------+
|       health_beauty|   1258681.34|    9670|    130.16|              9.26|
|       watches_gifts|   1205005.68|    5991|    201.14|              8.87|
|      bed_bath_table|   1036988.68|   11115|      93.3|              7.63|
|      sports_leisure|    988048.97|    8641|    114.34|              7.27|
|computers_accesso...|    911954.32|    7827|    116.51|              6.71|
|     furniture_decor|    729762.49|    8334|     87.56|              5.37|
|          cool_stuff|    635290.85|    3796|    167.36|              4.67|
|          housewares|    632248.66|    6964|     90.79|              4.65|
|                auto|    592720.11|    4235|    139.96|              4.36|
|        garden_tools|    485256.46|    4347|    111.63|              3.57|
|           

## Sauvegarde en zone gold

In [24]:
df_gold_revenue_delivered.write.mode("overwrite").parquet("../data/gold/account/revenue_delivered/")
df_gold_revenue_in_transit.write.mode("overwrite").parquet("../data/gold/account/revenue_in_transit/")
df_gold_revenue_excluded.write.mode("overwrite").parquet("../data/gold/account/revenue_excluded/")
df_monthly_revenue.write.mode("overwrite").parquet("../data/gold/account/monthly_revenue/")
df_gold_payment_breakdown.write.mode("overwrite").parquet("../data/gold/account/payment_breakdown/")
df_installments_by_type.write.mode("overwrite").parquet("../data/gold/account/installments_by_type/")
df_payments_by_customer_state.write.mode("overwrite").parquet("../data/gold/account/payments_by_customer_state/")
df_payments_by_customer_region.write.mode("overwrite").parquet("../data/gold/account/payments_by_customer_region/")
df_revenue_by_seller_state.write.mode("overwrite").parquet("../data/gold/account/revenue_by_seller_state/")
df_top_sellers.write.mode("overwrite").parquet("../data/gold/account/top_sellers/")
df_revenue_by_category.write.mode("overwrite").parquet("../data/gold/account/revenue_by_category/")